# Trabalho — Implementação dos 4 Algoritmos de Classificação
## KNN, Naive Bayes, Árvore de Decisão e Regras

**Disciplina:** Inteligência Artificial  
**Dataset:** Pesquisa sobre Gripe  
**Aluno:** Henrique Pacheco Alves — R.A.: 23293915-2  
**Aluno:** ESOFT7S-N-A

---
## 1. Carregamento e Preparação dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

# ── Carregar dataset da pesquisa sobre Gripe ──
url = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv"
df = pd.read_csv(url)

# Renomear colunas para nomes simples
mapping = {
    "Carimbo de data/hora": "timestamp",
    "Você ficou gripado no ano passado ?": "gripe_ano_passado",
    "Você tomou vacina da gripe no ano passado?": "vacina",
    "  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ": "ambientes_cheios",
    "  Você viajou no ano passado mais de 100 km de distância?  ": "viajou",
    "  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ": "alergia",
    "Quantas horas você dormiu em média por noite no ano passado?": "horas_sono",
    "Você praticou atividade física no ano passado?": "exercicio",
    "Você se alimentou de forma balanceada no ano passado?": "alimentacao",
    "Em média, quantas vezes você lavou as mãos por dia no ano passado?": "lavagem_maos",
    "Na sua percepção, o seu nível de estresse no ano passado foi:": "estresse"
}

df = df.rename(columns=mapping).dropna()

# Remover coluna de timestamp (não é feature)
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

print(f"Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas")
print(f"\nColunas: {list(df.columns)}")
display(df.head())

In [ ]:
# ── Definir target e features ──
TARGET = 'gripe_ano_passado'

# Padronizar valores do target
df[TARGET] = df[TARGET].astype(str).str.strip()

print("Distribuição do target (gripe_ano_passado):")
print(df[TARGET].value_counts())
print(f"\nTipos dos atributos:")
print(df.dtypes)

In [ ]:
# ── Pré-processamento ──
# Identificar colunas categóricas e numéricas
FEATURES = [c for c in df.columns if c != TARGET]

# Codificar variáveis categóricas com LabelEncoder
label_encoders = {}
df_encoded = df.copy()

for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le
        print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Converter tudo para numérico
for col in FEATURES:
    df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')

df_encoded = df_encoded.dropna()
print(f"\nDataset após encoding: {df_encoded.shape[0]} registros")

In [ ]:
# ── Separação treino/teste ──
RANDOM_STATE = 42
TEST_SIZE = 0.20

X = df_encoded[FEATURES].values
y = df_encoded[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Guardar também o DataFrame original (não-encoded) para o algoritmo de regras
df_original = df.loc[df_encoded.index].copy()
idx_train, idx_test = train_test_split(
    df_original.index, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=df_encoded.loc[df_original.index, TARGET]
)
df_regras_train = df_original.loc[idx_train].copy()
df_regras_test = df_original.loc[idx_test].copy()

print(f"Treino: {len(X_train)} amostras")
print(f"Teste:  {len(X_test)} amostras")
print(f"\nDistribuição no treino: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Distribuição no teste:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

---
## 2. KNN (K-Nearest Neighbors)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Pipeline com escalonamento (essencial para KNN, que é baseado em distância)
K = 5
knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=K, weights='distance'))
])

knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print("="*60)
print(f"KNN (K={K}, weights=distance)")
print("="*60)
print(f"Acurácia: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"\nRelatório de classificação:")
print(classification_report(y_test, y_pred_knn,
      target_names=label_encoders[TARGET].classes_))

In [ ]:
# Matriz de confusão do KNN
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_knn,
    display_labels=label_encoders[TARGET].classes_,
    cmap='Blues', ax=ax
)
ax.set_title(f'KNN (K={K})')
plt.tight_layout()
plt.show()

In [ ]:
# Teste com diferentes valores de K
print("Teste de diferentes valores de K:")
print(f"{'K':>4s}  {'Acurácia':>10s}")
print("-" * 16)
for k in [1, 3, 5, 7, 9, 11]:
    model_k = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k, weights='distance'))
    ])
    model_k.fit(X_train, y_train)
    acc = accuracy_score(y_test, model_k.predict(X_test))
    print(f"{k:>4d}  {acc:>10.4f}")

---
## 3. Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print("="*60)
print("NAIVE BAYES (Gaussiano)")
print("="*60)
print(f"Acurácia: {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"\nRelatório de classificação:")
print(classification_report(y_test, y_pred_nb,
      target_names=label_encoders[TARGET].classes_))

In [ ]:
# Matriz de confusão do Naive Bayes
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_nb,
    display_labels=label_encoders[TARGET].classes_,
    cmap='Oranges', ax=ax
)
ax.set_title('Naive Bayes (Gaussiano)')
plt.tight_layout()
plt.show()

In [ ]:
# Probabilidades preditas pelo Naive Bayes para cada amostra de teste
probs = nb_model.predict_proba(X_test)
print("Exemplo das probabilidades preditas (primeiras 5 amostras de teste):")
print(f"Classes: {label_encoders[TARGET].classes_}")
for i in range(min(5, len(probs))):
    classe_real = label_encoders[TARGET].inverse_transform([y_test[i]])[0]
    classe_pred = label_encoders[TARGET].inverse_transform([y_pred_nb[i]])[0]
    print(f"  Amostra {i+1}: probs={np.round(probs[i], 4)} → previsto={classe_pred} (real={classe_real})")

---
## 4. Árvore de Decisão

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=RANDOM_STATE,
    min_samples_leaf=3
)

tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

print("="*60)
print(f"ÁRVORE DE DECISÃO (max_depth=4)")
print("="*60)
print(f"Acurácia: {accuracy_score(y_test, y_pred_tree):.4f}")
print(f"\nRelatório de classificação:")
print(classification_report(y_test, y_pred_tree,
      target_names=label_encoders[TARGET].classes_))

In [ ]:
# Visualização da árvore
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    tree_model,
    feature_names=FEATURES,
    class_names=label_encoders[TARGET].classes_,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title('Árvore de Decisão — Dataset Gripe', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Regras extraídas da árvore (formato texto)
print("Regras da Árvore de Decisão:")
print(export_text(tree_model, feature_names=FEATURES))

In [ ]:
# Importância dos atributos
importancias = pd.Series(tree_model.feature_importances_, index=FEATURES)
importancias = importancias.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot(kind='barh', color='forestgreen', ax=ax)
ax.set_title('Importância dos Atributos — Árvore de Decisão')
ax.set_xlabel('Importância')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusão da Árvore
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tree,
    display_labels=label_encoders[TARGET].classes_,
    cmap='Greens', ax=ax
)
ax.set_title('Árvore de Decisão')
plt.tight_layout()
plt.show()

---
## 5. Regras de Classificação (PRISM)

Implementação do algoritmo **PRISM** conforme ensinado na Aula 006.  
Para cada classe, gera regras adicionando condições que maximizam `p(classe | atributo=valor)` até a regra ser perfeita ou não haver mais atributos.

In [ ]:
# ── Implementação do PRISM ──

def prism(df_treino, target, atributos):
    """
    Implementação do algoritmo PRISM para gerar regras de classificação.
    Retorna uma lista de regras no formato:
        [(condições_dict, classe_predita, cobertura), ...]
    """
    regras = []
    classes = df_treino[target].unique()

    for classe_alvo in classes:
        E = df_treino.copy()

        while len(E[E[target] == classe_alvo]) > 0:
            # Criar nova regra com antecedente vazio
            condicoes = {}
            attrs_disponiveis = list(atributos)
            subset = E.copy()

            while True:
                # Verificar se a regra já é perfeita
                n_pos = len(subset[subset[target] == classe_alvo])
                n_total = len(subset)

                if n_total == 0 or n_pos == n_total:
                    break
                if len(attrs_disponiveis) == 0:
                    break

                # Calcular p(classe | attr=val) para cada par disponível
                melhor_prob = -1
                melhor_cob = -1
                melhor_attr = None
                melhor_val = None

                for attr in attrs_disponiveis:
                    for val in subset[attr].unique():
                        sub = subset[subset[attr] == val]
                        total = len(sub)
                        positivos = len(sub[sub[target] == classe_alvo])
                        prob = positivos / total if total > 0 else 0

                        # Escolher o que maximiza prob; desempate por cobertura
                        if (prob > melhor_prob) or \
                           (prob == melhor_prob and positivos > melhor_cob):
                            melhor_prob = prob
                            melhor_cob = positivos
                            melhor_attr = attr
                            melhor_val = val

                if melhor_attr is None:
                    break

                # Adicionar condição à regra
                condicoes[melhor_attr] = melhor_val
                subset = subset[subset[melhor_attr] == melhor_val].copy()
                attrs_disponiveis.remove(melhor_attr)

            # Salvar regra se cobre algum exemplo positivo
            n_pos_final = len(subset[subset[target] == classe_alvo])
            if n_pos_final > 0 and len(condicoes) > 0:
                regras.append((condicoes, classe_alvo, n_pos_final))
                # Remover exemplos cobertos pela regra
                mask = pd.Series(True, index=E.index)
                for attr, val in condicoes.items():
                    mask &= (E[attr] == val)
                E = E[~mask].copy()
            else:
                break  # Não conseguiu gerar regra útil

    return regras


def prism_predict(regras, df_teste, target, classe_default):
    """
    Classifica usando as regras geradas pelo PRISM.
    Se nenhuma regra se aplica, usa a classe default (majoritária).
    """
    predicoes = []
    for _, row in df_teste.iterrows():
        classe_pred = classe_default  # regra default
        for condicoes, classe, _ in regras:
            match = all(row[attr] == val for attr, val in condicoes.items())
            if match:
                classe_pred = classe
                break
        predicoes.append(classe_pred)
    return predicoes


print("Funções PRISM definidas com sucesso.")

In [ ]:
# ── Discretizar atributos numéricos para o PRISM (aceita apenas nominais) ──

def discretizar(df_in, colunas_num):
    """Discretiza colunas numéricas em faixas (baixo/médio/alto)."""
    df_out = df_in.copy()
    for col in colunas_num:
        if col in df_out.columns and pd.api.types.is_numeric_dtype(df_out[col]):
            df_out[col] = pd.qcut(
                df_out[col].rank(method='first'),
                q=3, labels=['baixo', 'medio', 'alto']
            )
    return df_out

# Identificar colunas numéricas
colunas_num = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
print(f"Colunas numéricas a discretizar: {colunas_num}")

df_disc_train = discretizar(df_regras_train, colunas_num)
df_disc_test = discretizar(df_regras_test, colunas_num)

# Padronizar strings
for col in df_disc_train.columns:
    df_disc_train[col] = df_disc_train[col].astype(str).str.strip()
    df_disc_test[col] = df_disc_test[col].astype(str).str.strip()

print(f"\nTreino discretizado: {len(df_disc_train)} registros")
display(df_disc_train.head())

In [ ]:
# ── Executar PRISM ──
FEATURES_REGRAS = [c for c in df_disc_train.columns if c != TARGET]

regras = prism(df_disc_train, TARGET, FEATURES_REGRAS)

print("="*60)
print(f"REGRAS DE CLASSIFICAÇÃO (PRISM) — {len(regras)} regras geradas")
print("="*60)
for i, (conds, classe, cob) in enumerate(regras, 1):
    conds_str = ' AND '.join([f"{a} = {v}" for a, v in conds.items()])
    print(f"  Regra {i}: IF {conds_str} THEN {TARGET} = {classe}  (cobertura: {cob})")

In [ ]:
# ── Classificar teste com PRISM ──
classe_default = df_disc_train[TARGET].value_counts().idxmax()
y_pred_regras_str = prism_predict(regras, df_disc_test, TARGET, classe_default)

y_test_regras_str = df_disc_test[TARGET].values

# Acurácia
acc_regras = accuracy_score(y_test_regras_str, y_pred_regras_str)

print("="*60)
print(f"AVALIAÇÃO — REGRAS (PRISM)")
print("="*60)
print(f"Classe default: {classe_default}")
print(f"Acurácia: {acc_regras:.4f}")
print(f"\nRelatório de classificação:")
print(classification_report(y_test_regras_str, y_pred_regras_str))

In [ ]:
# Matriz de confusão das Regras
fig, ax = plt.subplots(figsize=(5, 4))
labels_regras = sorted(df[TARGET].unique())
ConfusionMatrixDisplay.from_predictions(
    y_test_regras_str, y_pred_regras_str,
    display_labels=labels_regras,
    cmap='Purples', ax=ax
)
ax.set_title('Regras (PRISM)')
plt.tight_layout()
plt.show()

---
## 6. Comparação dos Algoritmos

In [ ]:
# ── Comparação final ──
resultados = pd.DataFrame({
    'Algoritmo': ['KNN (K=5)', 'Naive Bayes', 'Árvore de Decisão', 'Regras (PRISM)'],
    'Acurácia': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_tree),
        acc_regras
    ]
})

resultados = resultados.sort_values('Acurácia', ascending=False).reset_index(drop=True)

print("="*60)
print("COMPARAÇÃO FINAL DOS ALGORITMOS")
print("="*60)
display(resultados)

# Gráfico de barras
fig, ax = plt.subplots(figsize=(8, 4))
cores = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
bars = ax.bar(resultados['Algoritmo'], resultados['Acurácia'], color=cores)
ax.set_ylabel('Acurácia')
ax.set_title('Comparação de Acurácia entre Algoritmos')
ax.set_ylim(0, 1.05)
for bar, val in zip(bars, resultados['Acurácia']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrizes de confusão lado a lado ──
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

configs = [
    (y_test, y_pred_knn, 'KNN', 'Blues', label_encoders[TARGET].classes_),
    (y_test, y_pred_nb, 'Naive Bayes', 'Oranges', label_encoders[TARGET].classes_),
    (y_test, y_pred_tree, 'Árvore', 'Greens', label_encoders[TARGET].classes_),
    (y_test_regras_str, y_pred_regras_str, 'Regras (PRISM)', 'Purples', labels_regras),
]

for ax, (yt, yp, titulo, cmap, labels) in zip(axes, configs):
    ConfusionMatrixDisplay.from_predictions(
        yt, yp, display_labels=labels, cmap=cmap, ax=ax
    )
    ax.set_title(titulo)

plt.suptitle('Matrizes de Confusão — Todos os Algoritmos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Conclusão

Neste trabalho foram implementados e comparados os 4 algoritmos de classificação estudados na disciplina:

- **KNN**: classificador baseado em distância, com escalonamento via `StandardScaler`. Testados diferentes valores de K.
- **Naive Bayes (Gaussiano)**: classificador probabilístico que assume independência entre os atributos.
- **Árvore de Decisão**: modelo interpretável que gera regras hierárquicas de decisão. Visualizada a árvore e a importância dos atributos.
- **Regras (PRISM)**: implementação manual do algoritmo PRISM conforme ensinado em aula, gerando regras de classificação interpretáveis.

O dataset utilizado foi a **pesquisa sobre Gripe** sugerida pelo professor, com pré-processamento (encoding, discretização para o PRISM) e separação treino/teste (80/20) com estratificação.